# graficos para steramlit

In [2]:
pip install streamlit


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from datetime import datetime, timedelta
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
import gspread
from google.oauth2 import service_account
from googleapiclient.discovery import build
import requests
import gspread_dataframe as gd
import google.auth
import google.auth.transport.requests
import pandas as pd
import numpy as np
import io

from sqlalchemy.engine import create_engine
from sqlalchemy.dialects.oracle import (FLOAT,NUMBER,VARCHAR2,TIMESTAMP,DATE,VARCHAR)
from sqlalchemy import Float, Numeric
from sqlalchemy import delete, text
import plotly.express as pex

import plotly.io as pio
pio.renderers.default = 'browser'

## cargue de bases

In [ ]:
username = "SEMAFORIZACION"
password = "S3M4F0R0S2025"
host = "192.168.100.83:1521"
port = "1521"
sid = "GEOS"

dsn = f"oracle+oracledb://{username}:{password}@{host}:{port}/{sid}"
engine = create_engine(dsn, thick_mode={})

In [4]:
sem = 1
now = datetime.now()

day1=(now - timedelta(weeks=sem)).strftime("%d/%m/%y")
day2=(now + timedelta(days=1)).strftime("%d/%m/%y")

In [5]:
select_template = '''SELECT * FROM PLAN_HIST_SEMA WHERE "Tiempo" >= TO_DATE('{day_base}','DD/MM/RR') AND "Tiempo" <= TO_DATE('{day_fin}','DD/MM/RR') '''   #
query = select_template.format(day_base = day1, day_fin = day2)
Hist_sem = pd.read_sql(query, engine)
lista_ext = sorted(Hist_sem['Externo'].unique())
lista_ext = [np.int64(x) for x in lista_ext]

In [6]:
from datetime import timedelta
import pandas as pd
import numpy as np

# =====================================================
# FECHA Y HORA ACTUAL DE EJECUCIÓN
# =====================================================

ahora = pd.Timestamp.now().floor("S")
hoy = ahora.normalize()

# =====================================================
# PROCESAMIENTO DE PLANES
# =====================================================

data = []

for EXT in lista_ext:

    # -------------------------------------------------
    # FILTRAR Y ORDENAR
    # -------------------------------------------------

    planes_sel = Hist_sem[
        Hist_sem["Externo"] == EXT
    ].sort_values(
        by=["Referencia"],
        ascending=False
    )

    # -------------------------------------------------
    # INFORMACIÓN DE INICIO
    # -------------------------------------------------

    nuevo = (
        planes_sel[
            [
                "Referencia",
                "Externo",
                "Plan_ingresa",
                "Tiempo"
            ]
        ]
        .reset_index(drop=True)
    )

    inicio = pd.DataFrame(
        {
            "fecha_inicio": nuevo["Tiempo"].dt.date,
            "hora_inicio": nuevo["Tiempo"].dt.time
        }
    )

    nuevo = pd.concat(
        [nuevo, inicio],
        axis=1
    )

    # -------------------------------------------------
    # DETERMINAR FIN DEL ÚLTIMO PLAN
    # -------------------------------------------------

    ultima_fecha = planes_sel.iloc[0]["Tiempo"].normalize()

    # Si es el día actual → terminar en la hora actual
    if ultima_fecha == hoy:

        tiempo_fin = ahora

    # Si es un día histórico → terminar a las 23:59:59
    else:

        tiempo_fin = (
            planes_sel.iloc[0]["Tiempo"]
            .to_period("D")
            .to_timestamp(how="end")
            .round("1s")
            - timedelta(seconds=1)
        )

    add = pd.DataFrame(
        {
            "Plan_finaliza": [
                planes_sel.iloc[0]["Plan_ingresa"]
            ],
            "Tiempo": [
                tiempo_fin
            ]
        }
    )

    # -------------------------------------------------
    # INFORMACIÓN DE FINALIZACIÓN
    # -------------------------------------------------

    movido = planes_sel[
        [
            "Plan_finaliza",
            "Tiempo"
        ]
    ]

    prueba = (
        pd.concat(
            [add, movido],
            ignore_index=True
        )
        .set_axis(
            [
                "Plan_finaliza",
                "Tiempo_finaliza"
            ],
            axis=1
        )[:-1]
    )

    tiempos = pd.DataFrame(
        {
            "fecha_finaliza":
                prueba["Tiempo_finaliza"].dt.date,

            "hora_finaliza":
                prueba["Tiempo_finaliza"].dt.time
        }
    )

    prueba = pd.concat(
        [prueba, tiempos],
        axis=1
    )

    # -------------------------------------------------
    # UNIÓN DE INFORMACIÓN
    # -------------------------------------------------

    Union = pd.concat(
        [nuevo, prueba],
        axis=1
    )

    Union["plan_Final"] = np.where(
        Union["Plan_ingresa"]
        == Union["Plan_finaliza"],
        Union["Plan_ingresa"],
        0
    )

    # -------------------------------------------------
    # DIVIDIR REGISTROS QUE CRUZAN DÍA
    # -------------------------------------------------

    temp_df = []

    for row in Union.itertuples(index=False):

        if row.fecha_inicio != row.fecha_finaliza:

            lista1 = list(row)

            lista1[8] = lista1[4]
            lista1[9] = "23:59:59"

            temp_df.append(lista1)

            lista2 = list(row)

            lista2[4] = lista2[8]
            lista2[5] = "00:00:01"

            temp_df.append(lista2)

        else:

            temp_df.append(
                list(row)
            )

    # -------------------------------------------------
    # DATAFRAME FINAL
    # -------------------------------------------------

    df_planes = pd.DataFrame(
        temp_df,
        columns=Union.columns
    )

    # -------------------------------------------------
    # CONVERSIONES
    # -------------------------------------------------

    df_planes["fecha_inicio"] = pd.to_datetime(
        df_planes["fecha_inicio"]
    )

    df_planes["hora_inicio"] = pd.to_datetime(
        df_planes["hora_inicio"],
        format="%H:%M:%S"
    )

    df_planes["fecha_finaliza"] = pd.to_datetime(
        df_planes["fecha_finaliza"]
    )

    df_planes["hora_finaliza"] = pd.to_datetime(
        df_planes["hora_finaliza"],
        format="%H:%M:%S"
    )

    # -------------------------------------------------
    # DURACIÓN
    # -------------------------------------------------

    df_planes["duracion"] = (
        pd.Timestamp("now").normalize()
        +
        (
            df_planes["hora_finaliza"]
            -
            df_planes["hora_inicio"]
        )
    ).dt.time

    # -------------------------------------------------
    # FILTRO DE FECHA DE ANÁLISIS
    # -------------------------------------------------

    df_planes = df_planes[
        df_planes["fecha_inicio"]
        >= pd.to_datetime(
            day1,
            format="%d/%m/%y"
        )
    ]

    data.append(df_planes)

# =====================================================
# RESULTADO FINAL
# =====================================================

planes_ord = pd.concat(
    data,
    axis=0
).reset_index(drop=True)

/tmp/ipykernel_9045/1822917156.py:9: FutureWarning:

'S' is deprecated and will be removed in a future version, please use 's' instead.



In [7]:
deteccion_template = '''SELECT * FROM DET_RESUM_SEMA_15M WHERE "Tiempo" >= TO_DATE('{day_base}','DD/MM/RR') AND "Tiempo" <= TO_DATE('{day_fin}','DD/MM/RR') '''   #
query_det = deteccion_template.format(day_base = day1, day_fin = day2)
Hist_det = pd.read_sql(query_det, engine)

In [8]:
df = Hist_det

In [9]:
anexo_template = '''SELECT * FROM ESP_INT_SEMA '''   #
query_anx = anexo_template.format()
Anexo= pd.read_sql(query_anx, engine)

In [10]:
estado_template = '''SELECT * FROM EST_ACT_SEMA '''   #
query_est = estado_template.format()
estado= pd.read_sql(query_est, engine)

## GRÁFICAS

In [31]:
# =====================================================
# FILTROS
# =====================================================

ext_sel = "1165"
#acceso_sel = "1"
#sensor_sel = "W.1.1"

### Planes Semafóricos

In [16]:
import pandas as pd
import plotly.express as px

# =====================================================
# FILTRO
# =====================================================

df_filtrado = planes_ord[
    planes_ord["Externo"] == int(ext_sel)
].copy()

# =====================================================
# FORMATO PARA HOVER
# =====================================================

df_filtrado["FechaInicio"] = (
    df_filtrado["Tiempo"]
    .dt.strftime("%d/%m/%Y %H:%M:%S")
)

df_filtrado["FechaFin"] = (
    df_filtrado["Tiempo_finaliza"]
    .dt.strftime("%d/%m/%Y %H:%M:%S")
)

df_filtrado["Fecha"] = (
    df_filtrado["fecha_finaliza"]
    .dt.strftime("%d/%m/%Y")
)

# =====================================================
# PALETA OPERACIONAL SEMAFÓRICA
# =====================================================

color_map = {
    1: "#00FF00",   # Verde
    2: "#7FFF00",   # Verde amarillento
    3: "#FFFF00",   # Amarillo
    4: "#FFB000",   # Naranja claro
    5: "#FF7F00",   # Naranja
    6: "#FF4500",   # Naranja rojizo
    7: "#FF0000",   # Rojo
    8: "#FF00FF",   # Magenta
    9: "#00FFFF",   # Cian
    10: "#FFFFFF"   # Blanco
}

# =====================================================
# TIMELINE
# =====================================================

fig = px.timeline(
    df_filtrado,
    x_start="hora_inicio",
    x_end="hora_finaliza",
    y="Fecha",
    color="plan_Final",
    text="plan_Final",
    color_discrete_map=color_map,
    hover_data={
        "FechaInicio": True,
        "FechaFin": True,
        "duracion": True,
        "hora_inicio": False,
        "hora_finaliza": False,
        "Fecha": False
    }
)

# =====================================================
# ESTILO DE LAS BARRAS
# =====================================================

fig.update_traces(

    textposition="inside",

    textfont=dict(
        size=13,
        color="white"
    ),

    marker_line_color="white",

    marker_line_width=0.5,

    hovertemplate=
    "<b>Plan %{text}</b><br>" +
    "Inicio: %{customdata[0]}<br>" +
    "Fin: %{customdata[1]}<br>" +
    "Duración: %{customdata[2]}<extra></extra>"
)

# =====================================================
# ESTILO DASHBOARD OSCURO
# =====================================================

fig.update_layout(

    template="plotly_dark",

    height=900,

    paper_bgcolor="#111111",

    plot_bgcolor="#111111",

    title=dict(
        text=(
            f"Planes Semafóricos<br>"
            f"Externo {ext_sel}<br>"
            f"Periodo: {day1} a {day2}"
        ),
        x=0.5,
        font=dict(
            size=24,
            color="white"
        )
    ),

    xaxis_title="Hora del día",

    yaxis_title="Fecha",

    font=dict(
        color="white",
        size=13
    ),

    hoverlabel=dict(
        bgcolor="#222222",
        font_size=13,
        font_family="Arial"
    ),

    legend=dict(
        title="Plan",
        orientation="v",
        y=1,
        x=1.02,
        bgcolor="rgba(0,0,0,0)"
    ),

    margin=dict(
        l=90,
        r=140,
        t=100,
        b=60
    )
)

# =====================================================
# EJES
# =====================================================

fig.update_xaxes(

    title_font=dict(size=16),

    tickfont=dict(size=12),

    showgrid=True,

    gridcolor="#333333",

    tickformat="%H:%M",

    zeroline=False
)

fig.update_yaxes(

    title_font=dict(size=16),

    tickfont=dict(size=12),

    showgrid=True,

    gridcolor="#333333",

    autorange="reversed",

    zeroline=False
)

# =====================================================
# ANOTACIÓN RESUMEN
# =====================================================

fig.add_annotation(

    xref="paper",
    yref="paper",

    x=0.01,
    y=1.08,

    showarrow=False,

    align="left",

    text=(
        f"<b>Intersección:</b> {ext_sel}<br>"
        f"<b>Días analizados:</b> {df_filtrado['Fecha'].nunique()}<br>"
        f"<b>Cambios de plan:</b> {len(df_filtrado):,}"
    ),

    font=dict(
        size=13,
        color="white"
    ),

    bgcolor="rgba(30,30,30,0.9)",

    bordercolor="#666666",

    borderwidth=1
)

# =====================================================
# MOSTRAR
# =====================================================

fig.show()

### Detecciones

In [17]:
import pandas as pd
import plotly.graph_objects as go

# =====================================================
# DATOS
# =====================================================

df = Hist_det.copy()

df["Tiempo"] = pd.to_datetime(df["Tiempo"])
df["ext"] = df["ext"].astype(str)
df["Acceso"] = df["Acceso"].astype(str)

# =====================================================
# FILTRO SOLO EXTERNO
# =====================================================

df = df[
    df["ext"] == str(ext_sel)
].copy()

# =====================================================
# AGRUPAR POR TIEMPO Y ACCESO
# =====================================================

df_plot = (
    df.groupby(
        ["Tiempo", "Acceso"],
        as_index=False
    )["Deteccion"]
    .sum()
)

# =====================================================
# FIGURA
# =====================================================

fig = go.Figure()

for acceso in sorted(df_plot["Acceso"].unique()):

    temp = (
        df_plot[
            df_plot["Acceso"] == acceso
        ]
        .sort_values("Tiempo")
    )

    total_det = temp["Deteccion"].sum()

    fig.add_trace(

        go.Scatter(

            x=temp["Tiempo"],

            y=temp["Deteccion"],

            mode="lines",

            name=f"Acceso {acceso}<br>{total_det:,.0f} veh",

            line=dict(
                width=2
            ),

            hoverlabel=dict(
                bgcolor="#111111",
                bordercolor="#333333",
                font_size=11,
                font_family="Arial"
            ),

            hovertemplate=

            f"<b>ACCESO {acceso}</b><br>"
            "Detecciones: %{y}"
            "<extra></extra>"
        )
    )

# =====================================================
# LAYOUT
# =====================================================

fig.update_layout(

    template="plotly_dark",

    title=dict(
        text=f"Detecciones por Acceso - Ext {ext_sel}",
        x=0.5
    ),

    paper_bgcolor="#111111",

    plot_bgcolor="#111111",

    font=dict(
        color="white",
        size=13
    ),

    height=700,

    hovermode="x unified",

    showlegend=True,

    legend=dict(

        title="Accesos",

        x=0.99,
        y=0.99,

        xanchor="right",
        yanchor="top",

        bgcolor="rgba(0,0,0,0.70)",

        bordercolor="#888888",

        borderwidth=1,

        font=dict(
            size=11,
            color="white"
        )
    ),

    margin=dict(
        l=50,
        r=50,
        t=80,
        b=50
    )
)

# =====================================================
# EJES
# =====================================================

fig.update_xaxes(

    title="Tiempo",

    showgrid=True,

    gridwidth=0.5,

    rangeslider_visible=True
)

fig.update_yaxes(

    title="Detecciones",

    showgrid=True,

    gridwidth=0.5,

    rangemode="tozero"
)

# =====================================================
# MOSTRAR
# =====================================================

fig.show()

### Ocupación

In [18]:
import pandas as pd
import plotly.graph_objects as go

# =====================================================
# DATOS
# =====================================================

df = Hist_det.copy()

df["Tiempo"] = pd.to_datetime(df["Tiempo"])
df["ext"] = df["ext"].astype(str)
df["Acceso"] = df["Acceso"].astype(str)

# =====================================================
# FILTRO SOLO EXTERNO
# =====================================================

df = df[
    df["ext"] == str(ext_sel)
].copy()

# =====================================================
# AGRUPAR POR TIEMPO Y ACCESO
# =====================================================

df_plot = (
    df.groupby(
        ["Tiempo", "Acceso"],
        as_index=False
    )["Ocupacion"]
    .mean()
)

# =====================================================
# FIGURA
# =====================================================

fig = go.Figure()

for acceso in sorted(df_plot["Acceso"].unique()):

    temp = (
        df_plot[
            df_plot["Acceso"] == acceso
        ]
        .sort_values("Tiempo")
    )

    ocup_media = temp["Ocupacion"].mean()

    fig.add_trace(

        go.Scatter(

            x=temp["Tiempo"],

            y=temp["Ocupacion"],

            mode="lines",

            name=f"Acceso {acceso}<br>{ocup_media:.1f}%",

            line=dict(
                width=2
            ),

            hoverlabel=dict(
                bgcolor="#111111",
                bordercolor="#333333",
                font_size=11,
                font_family="Arial"
            ),

            hovertemplate=

            f"<b>ACCESO {acceso}</b><br>"

            "Ocupación: %{y:.1f}%"

            "<extra></extra>"
        )
    )

# =====================================================
# LAYOUT
# =====================================================

fig.update_layout(

    template="plotly_dark",

    title=dict(
        text=f"Ocupación por Acceso - Ext {ext_sel}",
        x=0.5
    ),

    paper_bgcolor="#111111",

    plot_bgcolor="#111111",

    font=dict(
        color="white",
        size=13
    ),

    height=700,

    hovermode="x unified",

    showlegend=True,

    legend=dict(

        title="Accesos",

        x=0.99,
        y=0.99,

        xanchor="right",
        yanchor="top",

        bgcolor="rgba(0,0,0,0.70)",

        bordercolor="#888888",

        borderwidth=1,

        font=dict(
            size=11,
            color="white"
        )
    ),

    margin=dict(
        l=50,
        r=50,
        t=80,
        b=50
    )
)

# =====================================================
# EJES
# =====================================================

fig.update_xaxes(

    title="Tiempo",

    showgrid=True,

    gridcolor="#333333",

    rangeslider_visible=True
)

fig.update_yaxes(

    title="Ocupación (%)",

    showgrid=True,

    gridcolor="#333333",

    rangemode="tozero"
)

# =====================================================
# MOSTRAR
# =====================================================

fig.show()

### Ocupación vs Detección

In [19]:
import math
import pandas as pd
import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =====================================================
# DATOS
# =====================================================

df = Hist_det.copy()

df["Tiempo"] = pd.to_datetime(df["Tiempo"])
df["ext"] = df["ext"].astype(str)
df["Acceso"] = df["Acceso"].astype(str)

df = df[
    df["ext"] == str(ext_sel)
].copy()

df = df.dropna(
    subset=[
        "Ocupacion",
        "Deteccion"
    ]
)

# =====================================================
# AGRUPAR SENSORES
# =====================================================

df_plot = (
    df.groupby(
        ["Tiempo", "Acceso"],
        as_index=False
    )
    .agg(
        {
            "Deteccion": "sum",
            "Ocupacion": "mean"
        }
    )
)

df_plot["FechaHora"] = (
    df_plot["Tiempo"]
    .dt.strftime("%d/%m/%Y %H:%M")
)

# =====================================================
# ACCESOS
# =====================================================

accesos = sorted(df_plot["Acceso"].unique())

n_accesos = len(accesos)

cols = min(4, n_accesos)

rows = math.ceil(n_accesos / 4)

# =====================================================
# FIGURA
# =====================================================

fig = make_subplots(

    rows=rows,

    cols=cols,

    subplot_titles=[
        f"Acceso {a}"
        for a in accesos
    ],

    horizontal_spacing=0.05,

    vertical_spacing=0.10
)

# =====================================================
# COLORES
# =====================================================

colores = [

    "#00FFFF",
    "#FFD700",
    "#FF7F50",
    "#7CFC00",

    "#FF69B4",
    "#9370DB",
    "#00CED1",
    "#FFA500"
]

# =====================================================
# LOOP ACCESOS
# =====================================================

for i, acceso in enumerate(accesos):

    row = i // 4 + 1
    col = i % 4 + 1

    temp = (
        df_plot[
            df_plot["Acceso"] == acceso
        ]
        .copy()
        .sort_values("Tiempo")
    )

    if len(temp) < 10:
        continue

    x = temp["Ocupacion"].values
    y = temp["Deteccion"].values

    # =================================================
    # MODELO CUADRÁTICO
    # =================================================

    coef = np.polyfit(x, y, 2)

    a, b, c = coef

    modelo = np.poly1d(coef)

    y_pred = modelo(x)

    ss_res = np.sum(
        (y - y_pred) ** 2
    )

    ss_tot = np.sum(
        (y - np.mean(y)) ** 2
    )

    r2 = 1 - (ss_res / ss_tot)

    x_fit = np.linspace(
        x.min(),
        x.max(),
        300
    )

    y_fit = modelo(x_fit)

    color = colores[
        i % len(colores)
    ]

    # =================================================
    # NUBE DE PUNTOS
    # =================================================

    fig.add_trace(

        go.Scatter(

            x=temp["Ocupacion"],

            y=temp["Deteccion"],

            mode="markers",

            showlegend=False,

            marker=dict(

                size=6,

                color=color,

                opacity=0.40
            ),

            customdata=np.stack(

                (
                    temp["FechaHora"],
                ),

                axis=-1
            ),

            hovertemplate=

            f"<b>EXT {ext_sel}</b><br><br>"

            f"<b>ACCESO {acceso}</b><br><br>"

            "%{customdata[0]}<br><br>"

            "Ocupación Promedio: %{x:.1f}%<br>"

            "Detecciones Totales: %{y}"

            "<extra></extra>"

        ),

        row=row,

        col=col
    )

    # =================================================
    # CURVA
    # =================================================

    fig.add_trace(

        go.Scatter(

            x=x_fit,

            y=y_fit,

            mode="lines",

            showlegend=False,

            line=dict(

                color=color,

                width=3
            ),

            hoverinfo="skip"

        ),

        row=row,

        col=col
    )

    # =================================================
    # TEXTO MODELO
    # =================================================

    ecuacion = (
        f"y={a:.4f}x²"
        f" + {b:.4f}x"
        f" + {c:.4f}"
    )

    fig.add_annotation(

        x=0.02,

        y=0.98,

        xref=f"x{i+1} domain" if i > 0 else "x domain",

        yref=f"y{i+1} domain" if i > 0 else "y domain",

        text=(

            f"<b>{ecuacion}</b><br>"

            f"R² = {r2:.3f}<br>"

            f"N = {len(temp):,}<br>"

            f"Det = {temp['Deteccion'].sum():,.0f}"

        ),

        showarrow=False,

        align="left",

        font=dict(

            size=10,

            color="white"
        ),

        bgcolor="rgba(0,0,0,0.75)",

        bordercolor=color,

        borderwidth=1
    )

# =====================================================
# LAYOUT
# =====================================================

fig.update_layout(

    template="plotly_dark",

    width=min(
        1800,
        cols * 450
    ),

    height=rows * 450,

    paper_bgcolor="#111111",

    plot_bgcolor="#111111",

    title=dict(

        text=f"Detecciones Totales vs Ocupación Promedio por Acceso - Ext {ext_sel}",

        x=0.5
    ),

    showlegend=False,

    margin=dict(

        l=40,

        r=40,

        t=80,

        b=40
    )
)

# =====================================================
# EJES
# =====================================================

fig.update_xaxes(

    title="Ocupación Promedio (%)",

    gridcolor="#333333",

    zeroline=False
)

fig.update_yaxes(

    title="Detecciones Totales",

    gridcolor="#333333",

    zeroline=False
)

# =====================================================
# MOSTRAR
# =====================================================

fig.show()

## mapas a poner

### 1 Mapa Sistema Semaforización Inteligente

In [20]:
import pandas as pd
import plotly.express as px

# =====================================================
# DATOS
# =====================================================

df = Anexo.copy()

df = df.dropna(
    subset=[
        "latitud",
        "longitud"
    ]
)

# =====================================================
# INDICADORES
# =====================================================

total_inter = len(df)

total_zonas = df["ZONA AUTO"].nunique()

total_wide = df["Wide"].sum()

total_narrow = df["Narrow"].sum()

fecha_inv = df["FECHA DE INSTALACION"].max()

# =====================================================
# PALETA POR ZONA AUTO
# =====================================================

zonas = sorted(
    df["ZONA AUTO"]
    .astype(str)
    .unique()
)

paleta = px.colors.qualitative.Dark24

color_map = {
    zona: paleta[i % len(paleta)]
    for i, zona in enumerate(zonas)
}

# =====================================================
# MAPA
# =====================================================

fig = px.scatter_mapbox(

    df,

    lat="latitud",

    lon="longitud",

    color="ZONA AUTO",

    color_discrete_map=color_map,

    zoom=11.3,

    height=900,

    custom_data=[

        "externo",

        "DIRECCION CORTA",

        "localidad",

        "ZONA PLANEAMIENTO",

        "REFERENCIA EQUIPO",

        "# INTERSECCIONES POR EQUIPO",

        "FECHA DE INSTALACION",

        "ZONA AUTO",

        "funcionamiento",

        "TIPO DE INTERSECCION",

        "Grupos Vehiculares",

        "Grupos Peatonales",

        "Wide",

        "Grupos Wide",

        "Narrow",

        "grupos Narrow",

        "Shut Down",

        "LINK CONFIG VD",

        "LINK DATEM",

        "LINK REPOSITORIO",

        "LINK ESQUEMAS",

        "LINK AUTOMATICO",

        "PRIORIDAD DE ATENCION"
    ]
)

# =====================================================
# HOVER
# =====================================================

fig.update_traces(

    marker=dict(
        size=9,
        opacity=0.75
    ),
    
    hoverlabel=dict(
        bgcolor="rgba(20,20,20,0.65)",
        font_size=11,
        font_family="Arial"
    ),

    hovertemplate=

    "<b>EXTERNO:</b> %{customdata[0]}<br><br>"

    "<b>DIRECCIÓN:</b> %{customdata[1]}<br>"

    "<b>LOCALIDAD:</b> %{customdata[2]}<br>"

    "<b>ZONA PLANEAMIENTO:</b> %{customdata[3]}<br>"

    "<b>ZONA AUTO:</b> %{customdata[7]}<br><br>"

    "<b>EQUIPO:</b> %{customdata[4]}<br>"

    "<b>INTERSECCIONES EQUIPO:</b> %{customdata[5]}<br>"

    "<b>INSTALACIÓN:</b> %{customdata[6]}<br><br>"

    "<b>FUNCIONAMIENTO:</b> %{customdata[8]}<br>"

    "<b>TIPO INTERSECCIÓN:</b> %{customdata[9]}<br><br>"

    "<b>GRUPOS VEHICULARES:</b> %{customdata[10]}<br>"

    "<b>GRUPOS PEATONALES:</b> %{customdata[11]}<br><br>"

    "<b>WIDE:</b> %{customdata[12]}<br>"

    "<b>GRUPOS WIDE:</b> %{customdata[13]}<br><br>"

    "<b>NARROW:</b> %{customdata[14]}<br>"

    "<b>GRUPOS NARROW:</b> %{customdata[15]}<br><br>"

    "<b>SHUTDOWN:</b> %{customdata[16]}<br>"

    "<b>PRIORIDAD:</b> %{customdata[22]}<br><br>"

    "<i>Click para acceder a documentación</i>"

    "<extra></extra>"
)

# =====================================================
# LAYOUT
# =====================================================

fig.update_layout(

    mapbox_style="carto-darkmatter",

    mapbox=dict(

        center=dict(
            lat=4.65,
            lon=-74.10
        ),

        zoom=11.3,

        bearing=100,

        pitch=0
    ),

    dragmode="pan",

    showlegend=False,

    paper_bgcolor="#111111",

    plot_bgcolor="#111111",

    font=dict(
        color="white",
        size=13
    ),

    annotations=[

        dict(

            x=0.99,
            y=0.01,

            xref="paper",
            yref="paper",

            xanchor="right",
            yanchor="bottom",

            text=(

                "<b>Inventario Red</b><br><br>"

                f"{total_inter:,} intersecciones<br><br>"

                f"{total_zonas:,} zonas auto<br><br>"

                f"Wide : {total_wide:,}<br><br>"

                f"Narrow : {total_narrow:,}<br><br>"

                "<b>Última instalación</b><br>"

                f"{fecha_inv.strftime('%Y-%m-%d')}"
            ),

            showarrow=False,

            align="left",

            font=dict(
                size=16,
                color="white"
            ),

            bgcolor="rgba(0,0,0,0.70)",

            bordercolor="#888888",

            borderwidth=1,

            borderpad=8
        )
    ],

    margin=dict(
        l=0,
        r=0,
        t=0,
        b=0
    )
)

# =====================================================
# CONFIGURACIÓN
# =====================================================

config = {

    "scrollZoom": True,

    "displayModeBar": True,

    "doubleClick": "reset",

    "showTips": False
}

# =====================================================
# MOSTRAR
# =====================================================

fig.show(config=config)

/tmp/ipykernel_9045/3711658796.py:52: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



### 2. Mapa Estados Concert

In [21]:
import pandas as pd
import plotly.express as px

# =====================================================
# DATOS
# =====================================================

df = estado.copy()

df = df.dropna(
    subset=[
        "latitud",
        "longitud"
    ]
)

# =====================================================
# FECHA DEL ESTADO
# =====================================================

fecha_estado = df["fecha"].max()

# =====================================================
# PALETA POR ESTADO
# =====================================================

color_map = {
    "Operando": "#00FF00",
    "EN SERVICIO": "#00FF00",

    "Fuera de servicio": "#FF0000",
    "FUERA DE SERVICIO": "#FF0000",

    "Destello": "#FFD700",
    "DESTELLO": "#FFD700",

    "Mantenimiento": "#FF8C00",
    "MANTENIMIENTO": "#FF8C00",

    "Apagado": "#FF1493",
    "APAGADO": "#FF1493",

    "Operacion sin conexion": "#00BFFF"
}

# =====================================================
# MAPA
# =====================================================

fig = px.scatter_mapbox(
    df,

    lat="latitud",
    lon="longitud",

    color="estado",

    color_discrete_map=color_map,

    custom_data=[
        "externo",
        "direccion",
        "zona_auto",
        "localidad",
        "equipo",
        "shutdown",
        "estado",
        "operacion",
        "num_wide",
        "num_narrow"
    ],

    zoom=11.3,

    height=900
)

# =====================================================
# HOVER
# =====================================================

fig.update_traces(

    marker=dict(
        size=8,
        opacity=0.85
    ),

    hoverlabel=dict(
        bgcolor="#222222",
        font_size=12,
        font_family="Arial"
    ),

    hovertemplate=
    "<b>Externo:</b> %{customdata[0]}<br>" +

    "<b>Dirección:</b> %{customdata[1]}<br>" +

    "<b>Zona:</b> %{customdata[2]}<br>" +

    "<b>Localidad:</b> %{customdata[3]}<br>" +

    "<b>Equipo:</b> %{customdata[4]}<br>" +

    "<b>Shutdown:</b> %{customdata[5]}<br>" +

    "<b>Estado:</b> %{customdata[6]}<br>" +

    "<b>Operación:</b> %{customdata[7]}<br>" +

    "<b>Wide:</b> %{customdata[8]}<br>" +

    "<b>Narrow:</b> %{customdata[9]}" +

    "<extra></extra>"
)

# =====================================================
# LAYOUT
# =====================================================

fig.update_layout(

    mapbox_style="carto-darkmatter",

    mapbox=dict(

        center=dict(
            lat=4.65,
            lon=-74.10
        ),

        zoom=11.3,

        bearing=100,

        pitch=0
    ),

    dragmode="pan",

    paper_bgcolor="#111111",

    plot_bgcolor="#111111",

    font=dict(
        color="white",
        size=13
    ),

    annotations=[

        dict(

            x=0.99,
            y=0.15,

            xref="paper",
            yref="paper",

            xanchor="right",
            yanchor="bottom",

            text=(

                "<b>Fecha estado</b><br>" +

                pd.to_datetime(
                    fecha_estado
                ).strftime(
                    "%Y-%m-%d %H:%M"
                )
            ),

            showarrow=False,

            align="center",

            font=dict(
                size=16,
                color="white"
            ),

            bgcolor="rgba(0,0,0,0.70)",

            bordercolor="#888888",

            borderwidth=1,

            borderpad=6
        )
    ],

    legend=dict(

        title="Estado",

        x=0.99,
        y=0.01,

        xanchor="right",
        yanchor="bottom",

        orientation="v",

        bgcolor="rgba(0,0,0,0.70)",

        bordercolor="#888888",

        borderwidth=1,

        font=dict(
            size=11,
            color="white"
        )
    ),

    margin=dict(
        l=0,
        r=0,
        t=0,
        b=0
    )
)

# =====================================================
# CONFIGURACIÓN INTERACTIVA
# =====================================================

config = {

    "scrollZoom": True,

    "displayModeBar": True,

    "doubleClick": "reset",

    "showTips": False
}

# =====================================================
# MOSTRAR
# =====================================================

fig.show(config=config)

/tmp/ipykernel_9045/1446855908.py:50: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



### 3 Mapa Atención de Novedades

In [22]:
import pandas as pd
import numpy as np

df_inc = estado.copy()

# ==========================================
# INCIDENTES ACTIVOS
# ==========================================

incidentes = df_inc[
    df_inc["id_de_solicitud"].notna()
].copy()

# ==========================================
# TIEMPO EN HORAS
# ==========================================

incidentes["horas_atencion"] = (
    pd.to_timedelta(
        incidentes["tiempo_transcurrido"]
    )
    .dt.total_seconds()
    / 3600
)

# ==========================================
# GRUPOS DE TIEMPO
# ==========================================

incidentes["grupo_tiempo"] = pd.cut(

    incidentes["horas_atencion"],

    bins=[0,1,3,5,9999],

    labels=[
        "<1 hora",
        "1-3 horas",
        "3-5 horas",
        ">5 horas"
    ],

    include_lowest=True
)

# ==========================================
# KPIs
# ==========================================

total_incidentes = len(incidentes)

promedio_atencion = (
    incidentes["horas_atencion"]
    .mean()
)

estado_count = (
    df_inc["estado_de_la_interseccion"]
    .value_counts()
)

In [39]:
# =====================================================
# MAPA SEGUIMIENTO ATENCIÓN SEMA
# =====================================================
import pandas as pd
import numpy as np
import plotly.express as px
# =====================================================
# DATOS
# =====================================================
df_inc = estado.copy()
# =====================================================
# INCIDENTES ACTIVOS
# =====================================================
incidentes = df_inc[
    df_inc["id_de_solicitud"].notna()
].copy()
# =====================================================
# TIEMPO DE ATENCIÓN
# =====================================================
if len(incidentes) > 0:
    incidentes["horas_atencion"] = (
        pd.to_timedelta(
            incidentes["tiempo_transcurrido"]
        )
        .dt.total_seconds()
        / 3600
    )
    promedio_atencion = (
        incidentes["horas_atencion"]
        .mean()
    )
else:
    promedio_atencion = 0
# =====================================================
# KPIs
# =====================================================
total_incidentes = len(incidentes)
estado_count = (
    df_inc["estado_de_la_interseccion"]
    .value_counts()
)
# =====================================================
# INCIDENTE ACTIVO
# =====================================================
df_inc["incidente_activo"] = np.where(
    df_inc["id_de_solicitud"].notna(),
    "SI",
    "NO"
)
# =====================================================
# TAMAÑO DE PUNTOS
# =====================================================
df_inc["tamano"] = np.where(
    df_inc["incidente_activo"] == "SI",
    18,
    4
)
# =====================================================
# COLORES
# =====================================================
color_map = {
    "EN SERVICIO": "#00FF66",
    "FUERA DE SERVICIO": "#FF3333",
    "DESTELLO": "#FFD700",
    "MANTENIMIENTO": "#FF8C00",
    "APAGADO": "#FF1493"
}
# =====================================================
# MAPA
# =====================================================
fig_mapa = px.scatter_mapbox(
    df_inc,
    lat="latitud",
    lon="longitud",
    color="estado_de_la_interseccion",
    color_discrete_map=color_map,
    size="tamano",
    size_max=18,
    custom_data=[
        "externo",
        "direccion",
        "estado_de_la_interseccion",
        "fecha",
        "causa",
        "id_de_solicitud",
        "tiempo_transcurrido"
    ],
    height=700
)
# =====================================================
# HOVER
# =====================================================
fig_mapa.update_traces(
    marker=dict(
        opacity=0.60
    ),

    hoverlabel=dict(
        bgcolor="rgba(20,20,20,0.65)",
        font_size=11,
        font_family="Arial"
    ),
    hovertemplate=
    "<b>Externo:</b> %{customdata[0]}<br>" +
    "<b>Dirección:</b> %{customdata[1]}<br><br>" +
    "<b>Estado:</b> %{customdata[2]}<br><br>" +
    "<b>Fecha:</b> %{customdata[3]}<br>" +
    "<b>Causa:</b> %{customdata[4]}<br>" +
    "<b>Solicitud:</b> %{customdata[5]}<br>" +
    "<b>Tiempo:</b> %{customdata[6]}" +
    "<extra></extra>"
)

# =====================================================
# CAJA RESUMEN
# =====================================================

texto_resumen = (
    "<b>Seguimiento Atención</b><br><br>"
    f"Incidentes activos : {total_incidentes}<br><br>"
    f"Promedio atención : {promedio_atencion:.1f} h<br><br>"
)

for est, cant in estado_count.items():
    texto_resumen += (
        f"{est}: {cant}<br>"
    )

# =====================================================
# LAYOUT
# =====================================================

fig_mapa.update_layout(
    mapbox_style="carto-darkmatter",
    mapbox=dict(
        center=dict(
            lat=4.65,
            lon=-74.10
        ),
        zoom=11.3,
        bearing=100,
        pitch=0
    ),
    dragmode="pan",
    showlegend=True,
    legend=dict(
        title="Estado Intersección",
        x=0.99,
        y=0.32,
        xanchor="right",
        yanchor="bottom",
        bgcolor="rgba(0,0,0,0.70)",
        bordercolor="#888888",
        borderwidth=1,
        font=dict(
            size=11,
            color="white"
        )
    ),
    paper_bgcolor="#111111",
    plot_bgcolor="#111111",
    font=dict(
        color="white",
        size=13
    ),
    annotations=[
        dict(
            x=0.99,
            y=0.01,
            xref="paper",
            yref="paper",
            xanchor="right",
            yanchor="bottom",
            text=texto_resumen,
            showarrow=False,
            align="left",
            font=dict(
                size=12,
                color="white"
            ),
            bgcolor="rgba(0,0,0,0.70)",
            bordercolor="#888888",
            borderwidth=1,
            borderpad=8
        )
    ],
    margin=dict(
        l=0,
        r=0,
        t=0,
        b=0
    )
)

# =====================================================
# CONFIGURACIÓN
# =====================================================

config = {
    "scrollZoom": True,
    "displayModeBar": True,
    "doubleClick": "reset",
    "showTips": False
}

# =====================================================
# MOSTRAR
# =====================================================

fig_mapa.show(config=config)

/tmp/ipykernel_9045/2439411589.py:71: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [37]:
import pandas as pd
import numpy as np
import plotly.express as px

# ==========================
# DATOS
# ==========================

incidentes_donut = estado[estado["tiempo_transcurrido"].notna()].copy()

incidentes_donut["horas_atencion"] = (
    pd.to_timedelta(incidentes_donut["tiempo_transcurrido"])
    .dt.total_seconds()/3600
)

incidentes_donut["grupo_tiempo"] = pd.cut(
    incidentes_donut["horas_atencion"],
    bins=[0,1,3,5,np.inf],
    labels=["<1 hora","1-3 horas","3-5 horas",">5 horas"],
    include_lowest=True
)

# ==========================
# RESUMEN
# ==========================

donut_df = (
    incidentes_donut["grupo_tiempo"]
    .value_counts()
    .reset_index()
)

donut_df.columns=["grupo","cantidad"]

orden=["<1 hora","1-3 horas","3-5 horas",">5 horas"]

donut_df["grupo"]=pd.Categorical(
    donut_df["grupo"],
    categories=orden,
    ordered=True
)

donut_df=donut_df.sort_values("grupo")

# ==========================
# COLORES
# ==========================

color_tiempo={
    "<1 hora":"#00FF66",
    "1-3 horas":"#FFD700",
    "3-5 horas":"#FF8C00",
    ">5 horas":"#FF3333"
}

# ==========================
# DONUT
# ==========================

fig_donut=px.pie(
    donut_df,
    values="cantidad",
    names="grupo",
    hole=0.70,
    color="grupo",
    color_discrete_map=color_tiempo
)

fig_donut.update_traces(
    textposition="inside",
    textinfo="percent+label",
    hovertemplate="<b>%{label}</b><br>Cantidad: %{value}<br>Porcentaje: %{percent}<extra></extra>"
)

fig_donut.update_layout(
    title=dict(text="Tiempo de Atención",x=0.5),
    paper_bgcolor="#111111",
    plot_bgcolor="#111111",
    font=dict(color="white",size=12),
    legend=dict(
        orientation="h",
        y=-0.15,
        x=0.5,
        xanchor="center",
        bgcolor="rgba(0,0,0,0)"
    ),
    margin=dict(l=10,r=10,t=50,b=30),
    height=280
)

fig_donut.show()

In [38]:
causas_df = (
    incidentes["causa"]
    .value_counts()
    .reset_index()
)
causas_df.columns = [
    "causa",
    "cantidad"
]
fig_barras = px.bar(
    causas_df,
    x="cantidad",
    y="causa",
    orientation="h",
    text="cantidad"
)

fig_barras.update_layout(

    title="Causas de Incidentes",
    paper_bgcolor="#111111",
    plot_bgcolor="#111111",
    font=dict(color="white"),
    height=280,
    yaxis=dict(
        categoryorder="total ascending"
    )
)

fig_barras.show()

### 4 Mapa Detecciones SEMA

In [26]:
# =====================================================
# ÚLTIMO DÍA DISPONIBLE
# =====================================================

fecha_max = Hist_det["Fecha"].max()

det_ult = Hist_det[
    Hist_det["Fecha"] == fecha_max
].copy()

# =====================================================
# OCUPACIÓN PROMEDIO POR SENSOR
# =====================================================

ocup_sensor = (
    det_ult
    .groupby(
        ["ext", "Sensor"],
        as_index=False
    )
    .agg(
        Ocupacion_prom=("Ocupacion", "mean")
    )
)

# =====================================================
# OCUPACIÓN PROMEDIO POR EXTERNO
# =====================================================

ocup_ext = (
    ocup_sensor
    .groupby(
        "ext",
        as_index=False
    )
    .agg(
        Ocupacion=("Ocupacion_prom", "mean")
    )
)

# =====================================================
# DETECCIONES TOTALES POR EXTERNO
# =====================================================

det_ext = (
    det_ult
    .groupby(
        "ext",
        as_index=False
    )
    .agg(
        Detecciones=("Deteccion", "sum")
    )
)

# =====================================================
# UNIÓN
# =====================================================

mapa_det = det_ext.merge(
    ocup_ext,
    on="ext",
    how="left"
)

# =====================================================
# AJUSTAR TIPO DE EXTERNO
# =====================================================

mapa_det["ext"] = mapa_det["ext"].astype(str)
df["externo"] = df["externo"].astype(str)

# =====================================================
# AGREGAR INFORMACIÓN GEOGRÁFICA
# =====================================================

mapa_det = mapa_det.merge(
    df[
        [
            "externo",
            "direccion",
            "localidad",
            "zona_auto",
            "equipo",
            "operacion",
            "longitud",
            "latitud"
        ]
    ].drop_duplicates(
        subset="externo"
    ),
    left_on="ext",
    right_on="externo",
    how="left"
)

In [27]:
# =====================================================
# CLASIFICACIÓN
# =====================================================

def clasificar_ocupacion(x):

    if pd.isna(x):
        return "Sin datos"

    elif x <= 5:
        return "Sin datos"

    elif x < 50:
        return "Fluido"

    elif x < 85:
        return "Saturado"

    else:
        return "Congestionado"


mapa_det["categoria"] = (
    mapa_det["Ocupacion"]
    .apply(clasificar_ocupacion)
)

In [28]:
import numpy as np

mapa_det["tamano"] = np.sqrt(
    mapa_det["Detecciones"]
)

mapa_det["tamano"] = (
    mapa_det["tamano"]
    .clip(lower=3)
)

In [29]:
import plotly.express as px

# =====================================================
# COLORES
# =====================================================

color_map = {

    "Sin datos": "#808080",

    "Fluido": "#00FF66",

    "Saturado": "#FFD700",

    "Congestionado": "#FF3333"
}

fig = px.scatter_mapbox(

    mapa_det,

    lat="latitud",

    lon="longitud",

    color="categoria",

    color_discrete_map=color_map,

    size="tamano",

    size_max=30,

    custom_data=[

        "ext",

        "direccion",

        "localidad",

        "Detecciones",

        "Ocupacion",

        "categoria",

        "equipo",

        "operacion"
    ],

    zoom=11.3,

    height=900
)

# =====================================================
# HOVER
# =====================================================

fig.update_traces(

    marker=dict(
        opacity=0.85
    ),

    hoverlabel=dict(
        bgcolor="#222222",
        font_size=12
    ),

    hovertemplate=

    "<b>Externo:</b> %{customdata[0]}<br>" +

    "<b>Dirección:</b> %{customdata[1]}<br>" +

    "<b>Localidad:</b> %{customdata[2]}<br><br>" +

    "<b>Detecciones:</b> %{customdata[3]:,.0f}<br>" +

    "<b>Ocupación:</b> %{customdata[4]:.1f}%<br>" +

    "<b>Estado:</b> %{customdata[5]}<br><br>" +

    "<b>Equipo:</b> %{customdata[6]}<br>" +

    "<b>Operación:</b> %{customdata[7]}" +

    "<extra></extra>"
)

# =====================================================
# LAYOUT
# =====================================================

fig.update_layout(

    mapbox_style="carto-darkmatter",

    mapbox=dict(

        center=dict(
            lat=4.65,
            lon=-74.10
        ),

        zoom=11.3,

        bearing=100,

        pitch=0
    ),

    dragmode="pan",

    paper_bgcolor="#111111",

    plot_bgcolor="#111111",

    font=dict(
        color="white"
    ),

    annotations=[

        dict(

            x=0.99,
            y=0.15,

            xref="paper",
            yref="paper",

            xanchor="right",
            yanchor="bottom",

            text=(
                "<b>Fecha análisis</b><br>"
                f"{pd.to_datetime(fecha_max).strftime('%Y-%m-%d')}"
            ),

            showarrow=False,

            align="center",

            font=dict(
                size=16,
                color="white"
            ),

            bgcolor="rgba(0,0,0,0.70)",

            bordercolor="#888888",

            borderwidth=1,

            borderpad=6
        )
    ],

    legend=dict(

        title="Nivel de Ocupación",

        x=0.99,
        y=0.01,

        xanchor="right",
        yanchor="bottom",

        bgcolor="rgba(0,0,0,0.70)",

        bordercolor="#888888",

        borderwidth=1,

        font=dict(
            size=11,
            color="white"
        )
    ),

    margin=dict(
        l=0,
        r=0,
        t=0,
        b=0
    )
)

# =====================================================
# CONFIG
# =====================================================

config = {

    "scrollZoom": True,

    "displayModeBar": True,

    "doubleClick": "reset",

    "showTips": False
}

# =====================================================
# MOSTRAR
# =====================================================

fig.show(config=config)

/tmp/ipykernel_9045/4177182601.py:18: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



## Matrices

### Matriz Día × Hora

In [36]:
import pandas as pd
import plotly.graph_objects as go

# =====================================================
# DATOS
# =====================================================

df = Hist_det.copy()

df["Tiempo"] = pd.to_datetime(df["Tiempo"])
df["ext"] = df["ext"].astype(str)

df = df[
    df["ext"] == str(ext_sel)
].copy()

# =====================================================
# VARIABLES TEMPORALES
# =====================================================

dias = {
    0: "Lunes",
    1: "Martes",
    2: "Miércoles",
    3: "Jueves",
    4: "Viernes",
    5: "Sábado",
    6: "Domingo"
}

df["DiaSemana"] = df["Tiempo"].dt.dayofweek
df["DiaNombre"] = df["DiaSemana"].map(dias)
df["Hora"] = df["Tiempo"].dt.hour

# =====================================================
# MATRIZ DE DETECCIONES
# =====================================================

matriz = (
    df.groupby(
        ["Hora", "DiaNombre"],
        as_index=False
    )["Deteccion"]
    .sum()
)

pivot = matriz.pivot(
    index="Hora",
    columns="DiaNombre",
    values="Deteccion"
)

orden_dias = [
    "Lunes",
    "Martes",
    "Miércoles",
    "Jueves",
    "Viernes",
    "Sábado",
    "Domingo"
]

pivot = pivot.reindex(columns=orden_dias)

# Asegurar las 24 horas

pivot = pivot.reindex(
    index=range(24),
    fill_value=0
)

pivot = pivot.fillna(0)

# =====================================================
# ESCALA DE COLOR
# =====================================================

colorscale = [

    [0.00, "#1F3B2D"],

    [0.25, "#4E8F73"],

    [0.50, "#C9B458"],

    [0.75, "#C98A4A"],

    [1.00, "#B85A5A"]

]

# =====================================================
# HEATMAP
# =====================================================

fig = go.Figure(

    go.Heatmap(

        z=pivot.values,

        x=pivot.columns,

        y=[
            f"{h:02d}:00"
            for h in pivot.index
        ],

        text=pivot.values,

        texttemplate="%{text:,.0f}",

        textfont=dict(
            color="rgba(255,255,255,0.85)",
            size=11
        ),

        colorscale=colorscale,

        xgap=1,

        ygap=1,

        colorbar=dict(

            title=dict(
                text="Detecciones",
                side="right"
            ),

            tickfont=dict(
                size=11
            )
        ),

        hovertemplate=

        "<b>%{x}</b><br>" +

        "<b>Hora:</b> %{y}<br>" +

        "<b>Detecciones:</b> %{z:,.0f}" +

        "<extra></extra>"
    )
)

# =====================================================
# LAYOUT
# =====================================================

fig.update_layout(

    template="plotly_dark",

    title=dict(

        text=f"Mapa de Calor de Detecciones - Ext {ext_sel}",

        x=0.5,

        y=0.98,

        font=dict(
            size=22
        )
    ),

    paper_bgcolor="#111111",

    plot_bgcolor="#111111",

    font=dict(
        color="white",
        size=13
    ),

    height=750,

    margin=dict(

        l=70,

        r=70,

        t=120,

        b=40
    )
)

# =====================================================
# EJE X
# =====================================================

fig.update_xaxes(

    side="top",

    title="",

    tickfont=dict(

        size=13,

        color="white"
    ),

    showgrid=False
)

# =====================================================
# EJE Y
# =====================================================

fig.update_yaxes(

    title="",

    tickfont=dict(

        size=11,

        color="white"
    ),

    showgrid=False
)

# =====================================================
# MOSTRAR
# =====================================================

fig.show()